In [ ]:
import cv2
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import os
from tqdm import tqdm

# On désactive les warnings de thread pour KMeans
os.environ["OMP_NUM_THREADS"] = "1"

df_lip = Lip_products.copy()

def get_shade_and_centers(image_path, category_l3):
    """
    Extrait la couleur dominante et renvoie également tous les centres de clusters trouvés.
    Retourne: (best_rgb_list, all_centers_list_of_lists)
    """
    img = cv2.imread(image_path)
    # Important : retourner un tuple (None, None) pour maintenir la structure
    if img is None: return None, None
    
    # 1. Resize
    img = cv2.resize(img, (100, 100), interpolation=cv2.INTER_AREA)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # 2. PARAMÈTRES
    s_min = 25 
    v_min = 60
    if category_l3 == 'Lip Liner':
        s_min = 45 

    # 3. MASQUE
    mask = cv2.inRange(hsv, (0, s_min, v_min), (180, 255, 250))
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # 4. EXTRACTION
    valid_pixels = img_rgb[mask > 0]
    
    if len(valid_pixels) < 10:
        mask_rescue = cv2.inRange(hsv, (0, 10, 40), (180, 255, 250))
        valid_pixels = img_rgb[mask_rescue > 0]
        if len(valid_pixels) < 10:
            return None, None

    # 5. K-MEANS
    try:
        if len(valid_pixels) > 1000:
            indices = np.random.choice(len(valid_pixels), 1000, replace=False)
            pixels_to_cluster = valid_pixels[indices]
        else:
            pixels_to_cluster = valid_pixels

        kmeans = KMeans(n_clusters=3, n_init=3, random_state=42)
        kmeans.fit(pixels_to_cluster)
        centers = kmeans.cluster_centers_ # Ceci est un numpy array float
        
        # 6. SÉLECTION DU MEILLEUR (SATURATION)
        centers_hsv = cv2.cvtColor(np.uint8([centers]), cv2.COLOR_RGB2HSV)[0]
        best_idx = np.argmax(centers_hsv[:, 1])
        
        # Formatage pour la sortie
        best_rgb = centers[best_idx].astype(int).tolist()
        
        # On convertit TOUS les centres en liste d'entiers pour stockage facile (Parquet/JSON)
        all_centers = centers.astype(int).tolist()
        
        return best_rgb, all_centers
        
    except Exception as e:
        return None, None

# --- LANCEMENT FINAL ---
print("🚀 Démarrage de l'extraction étendue (RGB + Centers)...")

# On s'assure de travailler sur la copie
df_lip = df_lip.copy() 

tqdm.pandas()

# 1. On applique la fonction qui retourne un tuple
# On stocke temporairement dans une Series de tuples
temp_results = df_lip.progress_apply(
    lambda row: get_shade_and_centers(
        os.path.join("images", str(row['image_filename'])), 
        row['category_level_3_name']
    ), axis=1
)

# 2. On "dézippe" la Series de tuples en deux colonnes distinctes
# Si temp_results contient des (None, None), zip gère ça correctement
df_lip['rgb_extracted'], df_lip['kmeans_centers'] = zip(*temp_results)

# Statistiques de succès
success_count = df_lip['rgb_extracted'].notna().sum()
total = len(df_lip)
print(f"✅ Extraction terminée : {success_count}/{total} produits ({success_count/total:.1%})")

# Vérification visuelle rapide des premières lignes
print("\nExemple de données extraites :")
print(df_lip[['rgb_extracted', 'kmeans_centers']].head(3))

# Sauvegarde
df_lip.to_parquet("lip_products_final_colors_full.parquet")
print("💾 Fichier sauvegardé : lip_products_final_colors_full.parquet")